# Data

In [3]:
!pip install indic-nlp-library huggingface_hub pandas matplotlib numpy scipy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 127.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 15.9 MB/s eta 0:00:00


In [2]:
import indicnlp
import huggingface_hub
import pandas as pd
import matplotlib
import numpy as np
import scipy

print("indic-nlp-library:", getattr(indicnlp, "__version__", "version not exposed"))
print("huggingface_hub:", huggingface_hub.__version__)
print("pandas:", pd.__version__)
print("matplotlib:", matplotlib.__version__)
print("numpy:", np.__version__)
print("scipy:", scipy.__version__)

indic-nlp-library: 0.92
huggingface_hub: 1.7.1
pandas: 2.2.2
matplotlib: 3.10.0
numpy: 2.0.2
scipy: 1.16.3


In [1]:
!pip install fasttext


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.4/73.4 kB 7.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached pybind11-3.0.2-py3-none-any.whl.metadata (10 kB)
Using cached pybind11-3.0.2-py3-none-any.whl (310 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp312-cp312-linux_x86_64.whl size=4647419 sha256=3b37cbf468c8caf2e94d39b04f9d20bde6771587f4335d6df198832421712d13
  Stored in directory: /root/.cache/pip/wheels/20/27/95/a7baf1b435f1cbde017cabdf1e9688526d2b0e929255a359c6
Successfully built fasttext


In [4]:
import torch
import torch.nn as nn
import json
import pandas as pd
import numpy as np
from indicnlp.tokenize import indic_tokenize
import string
import spacy
nlp = spacy.load("en_core_web_sm")
import pickle
from indicnlp.tokenize import indic_tokenize
import fasttext.util
import fasttext
import math
from torch.utils.data import DataLoader, Dataset
# Download the Hindi word vectors
# fasttext.util.download_model('hi', if_exists='ignore')

from tqdm import tqdm
import torch.optim as optim
from torch.amp import autocast, GradScaler

In [5]:
import torch
import pandas as pd
import numpy as np
import spacy
import fasttext
import pkg_resources

print("torch:", torch.__version__)
print("pandas:", pd.__version__)
print("numpy:", np.__version__)
print("spacy:", spacy.__version__)

# fasttext version (more reliable this way)
print("fasttext:", pkg_resources.get_distribution("fasttext").version)

# indic-nlp-library version (best via pkg_resources)
print("indic-nlp-library:", pkg_resources.get_distribution("indic-nlp-library").version)

# tqdm version
print("tqdm:", pkg_resources.get_distribution("tqdm").version)

torch: 2.10.0+cu128
pandas: 2.2.2
numpy: 2.0.2
spacy: 3.8.11
fasttext: 0.9.3
indic-nlp-library: 0.92
tqdm: 4.67.3


/tmp/ipykernel_8680/3936933076.py:6: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


In [ ]:
with open('2_merged_train_only (1).json','r') as f:
  train_data=json.load(f)

In [ ]:
train_data

{'English-Bengali': {'Train': {'78683': {'source': 'Do not forget to visit the point where the Narmada flowing through the marble rocks interchanges its calmness and serenity into insouciance.',
    'target': 'এই জায়গাগুলো দেখতে ভুলো না যেখানে নর্মদা নদী মার্বেল পাথরের পাহাড়ের মধ্য দিয়ে প্রবাহিত হচ্ছে এবং নিজের শান্তি ও সৌন্দর্যকে অনাসক্তিতে পরিণত করছে।'},
   '78684': {'source': 'It is evident that the biggest cause of poverty is illiteracy .',
    'target': 'এই কথা স্পষ্ট যে দরিদ্রতার বড় কারণ হল অশিক্ষা ।'},
   '78685': {'source': 'The film was released theatrically on 12 April 2013.',
    'target': 'চলচ্চিত্রটি ২০১৩ সালের ১২ই এপ্রিল প্রেক্ষাগৃহে মুক্তি পায়।'},
   '78686': {'source': "is wyatt's birthday party at ten p. m.",
    'target': 'অনিমেষ এর জন্মদিনের পার্টি রাত দশটায়'},
   '78687': {'source': 'Apart from being used as an eatable, barley is also used in many other fields like industries and agriculture.',
    'target': 'খাদ্যদ্রব্য ছাড়াও যব আরো বিভিন্ন ক্ষেত্রে যেমন শিল্প ও

In [ ]:
source_sentences_train=[]
target_sentences_train=[]
id_train=[]

source_sentences_test=[]
target_sentences_test=[]
id_test=[]

In [ ]:
for language_pair, language_data in train_data.items():
    if(language_pair == "English-Hindi"):
      print(f"Language Pair: {language_pair}")
      for data_type, data_entries in language_data.items():
          print(f"  Data Type: {data_type}")
          for entry_id, entry_data in data_entries.items():
              source = entry_data["source"]
              target = entry_data["target"]
              if (data_type == "Test"):
                source_sentences_test.append(source)
                target_sentences_test.append(target)
                id_test.append(entry_id)
              else:
                source_sentences_train.append(source)
                target_sentences_train.append(target)
                id_train.append(entry_id)

Language Pair: English-Hindi
  Data Type: Train


In [ ]:
len(source_sentences_train)

92340

In [ ]:
source_sentences_train[80797]

'Somebody on this side, what will be my goal-test?'

In [ ]:
target_sentences_train[80796]

'रसोईघर की बत्तियाँ बंद करें'

In [ ]:
target_sentences_train[80797]

['इस पक्ष में कोई, मेरे लक्ष्य परीक्षण क्या होगा?']

In [ ]:
ee=target_sentences_train[90000]

In [ ]:
len(target_sentences_train)

92340

In [ ]:
len(id_train)

92340

In [ ]:
train_x={'English':source_sentences_train,'Hindi':target_sentences_train}

In [ ]:

df=pd.DataFrame(train_x)

In [ ]:
df

,English,Hindi
0,cancel everything on my calendar,मेरे कैलेंडर पर सब कुछ रद्द करें
1,Adrenal hormone levels are at their peak durin...,अधिवृक्क के हार्मोन का स्तर प्रातःकाल में अपने...
2,"Golden threads are obtained from Surat, the qu...","स्वर्ण धागे सूरत से प्राप्त होते हैं, जिनकी गु..."
3,Look for agglutination within 30 seconds.,30 सेकेण्ड के भीतर एग्लूटिनेशन देखें।
4,The non-pompousness and informality of their l...,उनके जीवन की आडंबरहीनता एवं अनौपचारिकता उनके स...
...,...,...
80792,"So, is it that this is the optimization proble...","तो, यह अनुकूलन समस्या है जिसमें हम रुचि रखते थे।"
80793,In this Masjid made with red stones there are ...,लाल पत्थरों से बनायी गयी इस मस्जिद में हिन्दू ...
80794,"He began to work on the movie on August 17, 20...","उन्होंने 17 अगस्त, 2010 को फिल्म पर काम करना श..."
80795,start a new shopping list,एक नई खरीदारी सूची शुरू करें


In [ ]:

def preprocess_and_remove_punctuation(sentence):
    sentence = ''.join([char for char in sentence if char not in string.punctuation and not char.isdigit()])
    return sentence

In [ ]:
def preprocess(sentences):
  tokenized_sentences=[]
  for sentence in sentences:
    doc=nlp(sentence)
    tokenized_sentences.append([token.text for token in doc])
  return tokenized_sentences

In [ ]:
english_train_tokens = preprocess(source_sentences_train)
english_test_tokens=preprocess(source_sentences_test)

In [ ]:
len(english_train_tokens)

92340

In [ ]:
len(english_test_tokens)

23085

In [ ]:
with open('english_train_tokens','wb') as f:
  pickle.dump(english_train_tokens,f)

with open('english_test_tokens','wb') as f:
  pickle.dump(english_test_tokens,f)

In [6]:
with open('/content/drive/MyDrive/english_train_tokens','rb') as f:
  english_train_tokens_1=pickle.load(f)

with open('/content/drive/MyDrive/english_test_tokens','rb') as f:
  english_test_tokens_1=pickle.load(f)

In [7]:
len(english_train_tokens_1)

92340

In [8]:
len(english_test_tokens_1)

23085

In [ ]:
def preprocess_hindi(sentences):
    tokenized_sentences = []
    for sentence in sentences:
        tokens = indic_tokenize.trivial_tokenize_indic(sentence)
        tokenized_sentences.append(tokens)
    return tokenized_sentences

In [ ]:
len(target_sentences_train)

92340

In [ ]:

target_sentences_train_cleaned = [
    s[0] if isinstance(s, list) and len(s) == 1 else s
    for s in target_sentences_train
]


def preprocess_hindi(sentences):
    tokenized_sentences = []
    for sentence in sentences:
        tokens = indic_tokenize.trivial_tokenize_indic(sentence)
        tokenized_sentences.append(tokens)
    return tokenized_sentences

# Tokenize
hindi_tokens = preprocess_hindi(target_sentences_train_cleaned)


In [ ]:
len(hindi_tokens)

92340

In [ ]:
with open('/content/drive/MyDrive/ff_hindi_train_tokens','wb') as f:
  pickle.dump(hindi_tokens,f)

In [10]:
with open('/content/drive/MyDrive/hindi_train_tokens','rb') as f:
  hindi_tokens_1=pickle.load(f)

In [ ]:
len(hindi_tokens_1)

92340

# Encode and padding

In [11]:
en_train=english_train_tokens_1
en_test=english_test_tokens_1
hi_train=hindi_tokens_1

In [12]:
def build_vocab(datasets, specials=["<PAD>", "<SOS>", "<EOS>", "<UNK>"]):
    vocab = set()
    for ds in datasets:
        for sent in ds:
            vocab.update(sent)

    vocab -= set(specials)
    idx2word = specials + sorted(list(vocab))
    word2idx = {w: i for i, w in enumerate(idx2word)}
    return idx2word, word2idx


In [13]:

en_index2word, en_word2index = build_vocab([en_train, en_test])

hi_index2word, hi_word2index = build_vocab([hi_train])


In [ ]:
len(en_index2word)

80335

In [14]:

def encode_and_pad_fixed(sentences, word2index, max_length, device):

    encoded_tensors = []

    for sent in sentences:
        encoded = [word2index["<SOS>"]] + \
                  [word2index.get(word, word2index["<UNK>"]) for word in sent] + \
                  [word2index["<EOS>"]]


        if len(encoded) > max_length:
            encoded = encoded[:max_length]

        else:
            encoded += [word2index["<PAD>"]] * (max_length - len(encoded))

        encoded_tensors.append(torch.tensor(encoded, dtype=torch.long))


    padded = torch.stack(encoded_tensors)
    return padded.to(device)


device='cuda' if torch.cuda.is_available() else 'cpu'

max_len = 50  # manually chosen
en_train_tensor = encode_and_pad_fixed(en_train, en_word2index, max_len, device)
hi_train_tensor = encode_and_pad_fixed(hi_train, hi_word2index, max_len, device)


print("English train shape:", en_train_tensor.shape)
print("Hindi train shape:", hi_train_tensor.shape)


English train shape: torch.Size([92340, 50])
Hindi train shape: torch.Size([92340, 50])


# Embeddings

In [ ]:
!wget https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.hi.300.bin.gz

--2026-03-17 06:35:55--  https://dl.fbaipublicfiles.com/fasttext/vectors-crawl/cc.hi.300.bin.gz
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 65.8.76.77, 65.8.76.47, 65.8.76.35, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|65.8.76.77|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4371554972 (4.1G) [application/octet-stream]
Saving to: ‘cc.hi.300.bin.gz’

cc.hi.300.bin.gz    100%[===================>]   4.07G  6.82MB/s    in 5m 31s  

2026-03-17 06:41:26 (12.6 MB/s) - ‘cc.hi.300.bin.gz’ saved [4371554972/4371554972]



In [ ]:
!gunzip cc.hi.300.bin.gz

In [15]:
def load_glove_embeddings(glove_path):
    embeddings_index = {}
    with open(glove_path, encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            coefs = np.asarray(values[1:], dtype='float32')
            embeddings_index[word] = coefs
    print(f"Loaded {len(embeddings_index)} word vectors from GloVe.")
    return embeddings_index

glove_path = '/content/drive/MyDrive/glove.6B.300d.txt'
glove_embeddings = load_glove_embeddings(glove_path)


Loaded 400000 word vectors from GloVe.


In [ ]:
model = fasttext.load_model('cc.hi.300.bin')

In [ ]:
vector = model.get_word_vector('भारत')

In [ ]:
vector.shape

(300,)

In [ ]:
embedding_dim = 300
vocab_size = len(hi_word2index)
vectors = []

for word, idx in hi_word2index.items():
    vectors.append(model.get_word_vector(word))


fas_hi_embeddings = torch.tensor(vectors, dtype=torch.float32)


torch.save(fas_hi_embeddings, "/content/drive/MyDrive/fas_hi_embeddings.pt")

/tmp/ipykernel_9707/671678335.py:9: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /pytorch/torch/csrc/utils/tensor_new.cpp:253.)
  fas_hi_embeddings = torch.tensor(vectors, dtype=torch.float32)


In [29]:
# Load back
save_path_hi = '/content/drive/MyDrive/fas_hi_embeddings.pt'
fas_hi_embeddings = torch.load(save_path_hi)

In [30]:
fas_hi_embeddings.shape

torch.Size([82593, 300])

In [31]:
len(hi_word2index)

82593

# encoding and masks

In [ ]:
def encode_and_pad(vocab, sent, max_length):
    sos = vocab["<SOS>"]
    eos = vocab["<EOS>"]
    pad = vocab["<PAD>"]
    unk = vocab["<UNK>"]


    encoded = [vocab.get(w, unk) for w in sent]


    if len(encoded) > max_length - 2:
        encoded = encoded[:max_length - 2]


    padded = [sos] + encoded + [eos]
    padded += [pad] * (max_length - len(padded))
    return padded


max_length_1 = 50


en_train_encoded = [encode_and_pad(en_word2index, s, max_length_1) for s in en_train]
hi_train_encoded = [encode_and_pad(hi_word2index, s, max_length_1) for s in hi_train]


# Model and training

In [25]:

class PositionalEncoding(nn.Module):
    def __init__(self, d_model: int, dropout: float = 0.1, max_len: int = 10_000):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10_000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)

        self.register_buffer("pe", pe)

    def forward(self, x):
        # x: (batch, seq_len, d_model)
        x = x + self.pe[:, :x.size(1)]
        return self.dropout(x)


In [26]:

class TransformerMT(nn.Module):
    def __init__(self, src_vocab_size, tgt_vocab_size,
                 src_embeddings, tgt_embeddings,
                 src_pad_id=0,tgt_pad_id=0, d_model=300, nhead=6,
                 num_encoder_layers=6, num_decoder_layers=6,
                 dim_feedforward=1024, dropout=0.1):
        super().__init__()
        self.src_pad_id = src_pad_id
        self.tgt_pad_id=tgt_pad_id
        self.d_model = d_model


        self.src_embed = nn.Embedding.from_pretrained(
            src_embeddings, freeze=False, padding_idx=src_pad_id
        )
        self.tgt_embed = nn.Embedding.from_pretrained(
            tgt_embeddings, freeze=False, padding_idx=tgt_pad_id
        )

        self.pos_enc = PositionalEncoding(self.d_model, dropout=dropout)

        self.transformer = nn.Transformer(
            d_model=d_model,
            nhead=nhead,
            num_encoder_layers=num_encoder_layers,
            num_decoder_layers=num_decoder_layers,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            batch_first=True  # (B, T, C)
        )

        self.generator = nn.Linear(d_model, tgt_vocab_size)

    def src_make_pad_mask(self, ids):
        return (ids == self.src_pad_id)  # (B, T)

    def tgt_make_pad_mask(self,ids):
      return (ids==self.tgt_pad_id)

    def make_causal_mask(self, size, device):
        mask = torch.triu(torch.full((size, size), float("-inf"), device=device), diagonal=1)
        return mask

    def forward(self, src_ids, tgt_in_ids):

        src_key_padding_mask = self.src_make_pad_mask(src_ids)
        tgt_key_padding_mask = self.tgt_make_pad_mask(tgt_in_ids)
        tgt_mask = self.make_causal_mask(tgt_in_ids.size(1), device=src_ids.device)


        src_emb = self.pos_enc(self.src_embed(src_ids) * math.sqrt(self.d_model))
        tgt_emb = self.pos_enc(self.tgt_embed(tgt_in_ids) * math.sqrt(self.d_model))


        out = self.transformer(
            src=src_emb,
            tgt=tgt_emb,
            src_key_padding_mask=src_key_padding_mask,
            tgt_key_padding_mask=tgt_key_padding_mask,
            memory_key_padding_mask=src_key_padding_mask,
            tgt_mask=tgt_mask,
        )
        return self.generator(out)

In [27]:

def build_embedding_matrix(word2idx, pretrained_source, embedding_dim=300, is_fasttext=False):
    vocab_size = len(word2idx)
    embedding_matrix = np.zeros((vocab_size, embedding_dim))


    limit = np.sqrt(6 / embedding_dim)

    for word, idx in word2idx.items():
        try:
            if is_fasttext:
                embedding_matrix[idx] = pretrained_source.get_word_vector(word)
            else:
                embedding_matrix[idx] = pretrained_source[word]
        except (KeyError, AttributeError):

            embedding_matrix[idx] = np.random.uniform(-limit, limit, embedding_dim)

    return torch.tensor(embedding_matrix, dtype=torch.float32)


In [32]:

src_embeddings = build_embedding_matrix(
    en_word2index, glove_embeddings, embedding_dim=300, is_fasttext=False
)


tgt_embeddings = build_embedding_matrix(
    hi_word2index, fas_hi_embeddings, embedding_dim=300, is_fasttext=True
)


model = TransformerMT(
    src_vocab_size=len(en_word2index),
    tgt_vocab_size=len(hi_word2index),
    src_embeddings=src_embeddings,
    tgt_embeddings=tgt_embeddings,
    src_pad_id=en_word2index["<PAD>"],
    tgt_pad_id=hi_word2index["<PAD>"]
)


In [ ]:
!pip install --upgrade torch

In [ ]:
import torch
print(torch.__version__)

2.10.0+cu128


In [ ]:
def train_model(model, train_loader, val_loader, pad_id, num_epochs=10, lr=1e-4, device="cuda"):
    device = torch.device(device if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    model = model.to(device)
    criterion = nn.CrossEntropyLoss(ignore_index=pad_id)
    optimizer = optim.Adam(model.parameters(), lr=lr)


    scaler = GradScaler(device.type)


    # scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    #     optimizer, mode="min", factor=0.5, patience=3, verbose=True
    # )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="min", factor=0.5, patience=3
    )

    for epoch in range(num_epochs):
        model.train()
        total_loss = 0


        train_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Train]", leave=False)
        for src, tgt in train_bar:
            src, tgt = src.to(device), tgt.to(device)

            tgt_inp = tgt[:, :-1]
            tgt_out = tgt[:, 1:]

            optimizer.zero_grad()

            with autocast(device_type=device.type, enabled=(device.type == "cuda")):
                logits = model(src, tgt_inp)
                loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))

            if device.type == "cuda":
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                optimizer.step()

            total_loss += loss.item()
            train_bar.set_postfix(loss=loss.item())

        avg_train_loss = total_loss / len(train_loader)


        model.eval()
        val_loss = 0
        val_bar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{num_epochs} [Val]", leave=False)
        with torch.no_grad():
            for src, tgt in val_bar:
                src, tgt = src.to(device), tgt.to(device)
                tgt_inp = tgt[:, :-1]
                tgt_out = tgt[:, 1:]

                with autocast(device_type=device.type, enabled=(device.type == "cuda")):
                    logits = model(src, tgt_inp)
                    loss = criterion(logits.reshape(-1, logits.size(-1)), tgt_out.reshape(-1))
                val_loss += loss.item()
                val_bar.set_postfix(loss=loss.item())

        avg_val_loss = val_loss / len(val_loader)

        print(f"Epoch {epoch+1}/{num_epochs} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")


        scheduler.step(avg_val_loss)


        save_path = f"/content/drive/MyDrive/frf_transformer_epoch{epoch+1}.pth"
        torch.save(model.state_dict(), save_path)
        print(f" Model saved at {save_path}")


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import autocast, GradScaler
from torch.utils.data import DataLoader, Dataset
import random


class TranslationDataset(Dataset):
    def __init__(self, src_sequences, tgt_sequences):
        self.src = src_sequences
        self.tgt = tgt_sequences

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return torch.tensor(self.src[idx], dtype=torch.long), torch.tensor(self.tgt[idx], dtype=torch.long)

def collate_fn(batch, src_pad_id, tgt_pad_id):

    src_batch, tgt_batch = zip(*batch)
    src_batch = nn.utils.rnn.pad_sequence(src_batch, batch_first=True, padding_value=src_pad_id)
    tgt_batch = nn.utils.rnn.pad_sequence(tgt_batch, batch_first=True, padding_value=tgt_pad_id)
    return src_batch, tgt_batch



src = en_train_encoded
tgt = hi_train_encoded


data = list(zip(src, tgt))
random.shuffle(data)
src, tgt = zip(*data)

split_idx = int(0.9 * len(src))
train_src, val_src = src[:split_idx], src[split_idx:]
train_tgt, val_tgt = tgt[:split_idx], tgt[split_idx:]


src_pad_id = en_word2index["<PAD>"]
tgt_pad_id = hi_word2index["<PAD>"]


train_dataset = TranslationDataset(train_src, train_tgt)
val_dataset = TranslationDataset(val_src, val_tgt)

train_loader = DataLoader(
    train_dataset, batch_size=32, shuffle=True,
    collate_fn=lambda b: collate_fn(b, src_pad_id, tgt_pad_id),
    num_workers=4, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=32, shuffle=False,
    collate_fn=lambda b: collate_fn(b, src_pad_id, tgt_pad_id),
    num_workers=4, pin_memory=True
)


train_model(
    model,
    train_loader,
    val_loader,
    pad_id=tgt_pad_id,
    num_epochs=10,
    lr=1e-4,
    device="cuda" if torch.cuda.is_available() else "cpu"
)

Using device: cuda


Epoch 1/10 [Train]:   0%|          | 0/2598 [00:00<?, ?it/s]/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
/usr/local/lib/python3.12/dist-packages/torch/nn/modules/activation.py:1336: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = F._canonical_mask(


Epoch 1/10 | Train Loss: 6.0853 | Val Loss: 5.3761
 Model saved at /content/drive/MyDrive/frf_transformer_epoch1.pth


Epoch 2/10 | Train Loss: 5.0798 | Val Loss: 4.9574
 Model saved at /content/drive/MyDrive/frf_transformer_epoch2.pth


Epoch 3/10 | Train Loss: 4.6545 | Val Loss: 4.6996
 Model saved at /content/drive/MyDrive/frf_transformer_epoch3.pth


...

Epoch 6/10 | Train Loss: 3.7362 | Val Loss: 4.1168
 Model saved at /content/drive/MyDrive/frf_transformer_epoch6.pth


Epoch 7/10 | Train Loss: 3.4871 | Val Loss: 4.0037
 Model saved at /content/drive/MyDrive/frf_transformer_epoch7.pth


Epoch 8/10 | Train Loss: 3.2649 | Val Loss: 3.8950
 Model saved at /content/drive/MyDrive/frf_transformer_epoch8.pth


Epoch 9/10 | Train Loss: 3.0598 | Val Loss: 3.8193
 Model saved at /content/drive/MyDrive/frf_transformer_epoch9.pth


Epoch 10/10 | Train Loss: 2.8673 | Val Loss: 3.7749
 Model saved at /content/drive/MyDrive/frf_transformer_epoch10.pth


# Testing

In [33]:
import pickle
with open('/content/drive/MyDrive/final_test_df','rb') as f:
  test_data=pickle.load(f)

In [34]:

device='cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
device

'cuda'

In [35]:
source_sentences_test=[]
target_sentences_test=[]
id_val=[]

In [36]:
for language_pair, language_data in test_data.items():
    if(language_pair == "English-Hindi"):
      print(f"Language Pair: {language_pair}")
      for data_type, data_entries in language_data.items():
          print(f"  Data Type: {data_type}")
          for entry_id, entry_data in data_entries.items():
              source = entry_data["source"]
              target = entry_data["target"]
              if (data_type == "Test"):
                source_sentences_test.append(source)
                target_sentences_test.append(target)
                id_val.append(entry_id)
              else:
                source_sentences_train.append(source)
                target_sentences_train.append(target)
                id_train.append(entry_id)

Language Pair: English-Hindi
  Data Type: Test


In [ ]:
len(source_sentences_test)

23085

In [ ]:
len(target_sentences_test)

23085

In [40]:
test_x={'English':source_sentences_test,'Hindi':target_sentences_test}

In [41]:

test_df=pd.DataFrame(test_x)

In [ ]:
test_df

,English,Hindi
0,And then we need to assure students that a com...,[और फिर हम छात्रों को विश्वास दिलाने की जरूरत ...
1,what is the address for the event scheduled on...,[1 जनवरी को आयोजित कार्यक्रम के लिए पता क्या है]
2,Indira Gandhi National Park is spread in an ar...,[इंदिरा गांधी राष्ट्रीय उद्यान कोयंबटूर जिले म...
3,Local musicians also attracted the King's atte...,[स्थानीय संगीतकारों ने भी राजा की ओर ध्यान आकर...
4,"Of course, do some more stuff on this.","[निश्चय ही, इस पर कुछ और काम करें।]"
...,...,...
23080,send an email to peter and ask him how is he n...,[पीटर को एक ई-मेल भेजें और उससे पूछें कि वह कै...
23081,"At Bose ' s initiative , the Government of Ori...",[बोस की पहल से उड़ीसा सरकार ने प्रोफेसर हल्डेन...
23082,"v.After initial opening, store H2O2 at 4oC in ...","[v. आरंभिक खोलने के बाद, एच2ओ2 को 4 डिग्री सेल..."
23083,Holding the claw of the left leg with both han...,[उत्सर्जन के बाद दोनों हाथों से बाएं पैर की पा...


In [ ]:
test_df['Hindi']

,Hindi
0,[और फिर हम छात्रों को विश्वास दिलाने की जरूरत ...
1,[1 जनवरी को आयोजित कार्यक्रम के लिए पता क्या है]
2,[इंदिरा गांधी राष्ट्रीय उद्यान कोयंबटूर जिले म...
3,[स्थानीय संगीतकारों ने भी राजा की ओर ध्यान आकर...
4,"[निश्चय ही, इस पर कुछ और काम करें।]"
...,...
23080,[पीटर को एक ई-मेल भेजें और उससे पूछें कि वह कै...
23081,[बोस की पहल से उड़ीसा सरकार ने प्रोफेसर हल्डेन...
23082,"[v. आरंभिक खोलने के बाद, एच2ओ2 को 4 डिग्री सेल..."
23083,[उत्सर्जन के बाद दोनों हाथों से बाएं पैर की पा...


In [ ]:

with open('/content/drive/MyDrive/english_train_tokens','rb') as f:
  english_tokens_1=pickle.load(f)

with open('/content/drive/MyDrive/english_test_tokens','rb') as f:
  english_test_1=pickle.load(f)

In [ ]:

with open('/content/drive/MyDrive/hindi_train_tokens','rb') as f:
  hindi_tokens_1=pickle.load(f)

In [ ]:
en_train=english_tokens_1
en_test=english_test_1
hi_train=hindi_tokens_1

In [ ]:
len(english_tokens_1)

92340

In [ ]:
len(english_test_1)

23085

In [ ]:
len(hindi_tokens_1)

92340

In [ ]:
len(en_word2index)

80335

In [ ]:
len(hi_word2index)

82593

In [ ]:

def encode_and_pad_fixed(sentences, word2index, max_length, device):

    encoded_tensors = []

    for sent in sentences:
        encoded = [word2index["<SOS>"]] + \
                  [word2index.get(word, word2index["<UNK>"]) for word in sent] + \
                  [word2index["<EOS>"]]

        if len(encoded) > max_length:
            encoded = encoded[:max_length]
        else:
            encoded += [word2index["<PAD>"]] * (max_length - len(encoded))

        encoded_tensors.append(torch.tensor(encoded, dtype=torch.long))
    padded = torch.stack(encoded_tensors)
    return padded.to(device)

max_len = 50  # manually chosen
en_test_tensor = encode_and_pad_fixed(english_test_1, en_word2index, max_len, device)


print("English test shape:", en_test_tensor.shape)


English test shape: torch.Size([23085, 50])


In [ ]:

def load_glove_embeddings(glove_path):
    embeddings_index = {}
    with open(glove_path, encoding='utf-8') as f:
        for line in f:
            values = line.split()
            word = values[0]
            coefs = np.asarray(values[1:], dtype='float32')
            embeddings_index[word] = coefs
    print(f"Loaded {len(embeddings_index)} word vectors from GloVe.")
    return embeddings_index

glove_path = '/content/drive/MyDrive/glove.6B.300d.txt'
glove_embeddings = load_glove_embeddings(glove_path)


Loaded 400000 word vectors from GloVe.


In [23]:
import torch

from tqdm import tqdm
import json


class TestDataset(Dataset):
    def __init__(self, src_data):
        self.src_data = src_data

    def __len__(self):
        return len(self.src_data)

    def __getitem__(self, idx):
        return torch.tensor(self.src_data[idx], dtype=torch.long)


def collate_fn_test(batch, pad_id):
    max_len = max(len(x) for x in batch)
    padded = [torch.cat([x, torch.full((max_len - len(x),), pad_id,dtype=torch.long, device=x.device)]) for x in batch]
    return torch.stack(padded, dim=0)


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

pad_id_src = en_word2index["<PAD>"]
pad_id_tgt = hi_word2index["<PAD>"]

model = TransformerMT(
    src_vocab_size=len(en_word2index),
    tgt_vocab_size=len(hi_word2index),
    src_embeddings=src_embeddings,
    tgt_embeddings=tgt_embeddings,
    src_pad_id=pad_id_src,
    tgt_pad_id=pad_id_tgt
).to(device)

checkpoint_path = "/content/drive/MyDrive/frf_transformer_epoch10.pth"
model.load_state_dict(torch.load(checkpoint_path, map_location=device))
print(f"Loaded checkpoint: {checkpoint_path}")


def greedy_decode(model, src, max_len=60, pad_id=0, device="cpu"):
    model.eval()
    batch_size = src.size(0)


    sos_id = 1
    tgt = torch.full((batch_size, 1), sos_id, dtype=torch.long, device=device)

    with torch.no_grad():
        for _ in range(max_len - 1):
            logits = model(src, tgt)   # (B, T, vocab_size)
            next_token = logits[:, -1, :].argmax(-1, keepdim=True)  # (B, 1)
            tgt = torch.cat([tgt, next_token], dim=1)

    return tgt  # (B, max_len)


test_dataset = TestDataset(en_test_tensor)
test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=lambda b: collate_fn_test(b, pad_id_src)
)

save_path = "/content/drive/MyDrive/nnn_est_predictions.json"

all_preds = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Decoding", unit="batch"):
        src_ids = batch.to(device)


        batch_preds = greedy_decode(
            model, src_ids, max_len=60, pad_id=pad_id_tgt, device=device
        )


        batch_preds = [seq.tolist() for seq in batch_preds]
        all_preds.extend(batch_preds)


with open(save_path, "w", encoding="utf-8") as f:
    json.dump(all_preds, f, ensure_ascii=False, indent=2)

print(f"Saved test predictions to {save_path}")


In [38]:
import json
with open("/content/drive/MyDrive/nnn_est_predictions.json", "r", encoding="utf-8") as f:
    preds = json.load(f)


In [17]:
!pip install sacrebleu rouge-score

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.3 MB/s eta 0:00:00
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=bf70488be5391011aa7cc7ec3c2e96bcca50d40c0741eafe572b08fbf715402e
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge-score


In [18]:
!pip install rouge

In [19]:
import sacrebleu
import pkg_resources

print("sacrebleu:", sacrebleu.__version__)
print("rouge-score:", pkg_resources.get_distribution("rouge-score").version)
print("rouge:", pkg_resources.get_distribution("rouge").version)

sacrebleu: 2.6.0
rouge-score: 0.1.2
rouge: 1.0.1


In [42]:
import json
from sacrebleu.metrics import BLEU, CHRF
from rouge import Rouge

# Load raw token-ID predictions
# with open("/content/drive/MyDrive/nnn_est_predictions.json", "r", encoding="utf-8") as f:
#     preds = json.load(f)

# Decode token IDs → Hindi strings
eos_id = hi_word2index["<EOS>"]
pad_id = hi_word2index["<PAD>"]
sos_id = hi_word2index["<SOS>"]

skip_ids = {sos_id, eos_id, pad_id}

decoded_preds = []
for seq in preds:
    tokens = []
    for tok_id in seq:
        if tok_id == eos_id:
            break
        if tok_id not in skip_ids:
            tokens.append(hi_index2word[tok_id])
    decoded_preds.append(" ".join(tokens))

# Clean reference strings
decoded_refs = [
    r[0] if isinstance(r, list) else r
    for r in test_df['Hindi']
]

# Compute metrics
refs_nested = [decoded_refs]

bleu_metric = BLEU()
chrf_metric = CHRF()

bleu = bleu_metric.corpus_score(decoded_preds, refs_nested)
chrf = chrf_metric.corpus_score(decoded_preds, refs_nested)

rouge = Rouge()
scores = rouge.get_scores(decoded_preds, decoded_refs, avg=True)

metrics = {
    "BLEU": bleu.score,
    "chrF": chrf.score,
    "ROUGE-1": scores["rouge-1"]["f"] * 100,
    "ROUGE-2": scores["rouge-2"]["f"] * 100,
    "ROUGE-L": scores["rouge-l"]["f"] * 100
}

print(metrics)

{'BLEU': 9.448198355791833, 'chrF': 31.478134674849905, 'ROUGE-1': 40.91412090337956, 'ROUGE-2': 15.23398518260228, 'ROUGE-L': 35.31283279789}
